<a href="https://colab.research.google.com/github/nagomi-tech/blog-aibeginner/blob/main/Colab/japanese_speech_emotion_emotion2vec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 日本語音声感情分析：emotion2vec+ 性能ベンチマーク（ファインチューニングなし）

これまでのVAD（連続値）ベースのモデル（audEERING製 wav2vec2）では、英語ですら声の抑揚だけから
感情を明確に分離できませんでした。実務者向けには「使えるかどうか」の判断が重要なので、
パラダイムをVADからカテゴリ分類に切り替え、多言語対応の **emotion2vec+**（Alibaba FunASR）を、
**ファインチューニングなしで日本語にそのまま使えるか**検証します。

## 使用するもの
- **モデル**：`iic/emotion2vec_plus_large`（FunASR経由）。9クラス分類：
  angry / disgusted / fearful / happy / neutral / other / sad / surprised / unknown
- **データ**：JVNV（日本語感情音声コーパス、Cao型ではなく4名の日本語ネイティブ俳優による収録）を
  Hugging Face上のミラー `asahi417/jvnv-emotional-speech-corpus` から取得。
  6感情（anger, disgust, fear, happy, sad, surprise）× 4話者、人間の実発話です。
- **ライセンス注意**：emotion2vec+はFunASR独自の"model-license"です。標準的なOSSライセンス
  （Apache/MIT等）ではないため、商用利用前に必ずライセンス原文を確認してください
  （https://github.com/alibaba-damo-academy/FunASR/blob/main/MODEL_LICENSE）。

## 検証の流れ
1. 環境セットアップ
2. emotion2vec+largeのロード
3. JVNVコーパスの読み込みとサンプリング
4. 推論の実行
5. 正解ラベルとの突き合わせ（Accuracy・混同行列）
6. 感情ごとの性能を可視化


## 1. 環境セットアップ

In [ ]:
!pip install -q -U funasr modelscope datasets soundfile pandas matplotlib japanize-matplotlib scikit-learn


In [ ]:
import os
import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
import japanize_matplotlib
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

os.makedirs("audio_samples", exist_ok=True)
os.makedirs("results", exist_ok=True)

print("セットアップ完了")


## 2. emotion2vec+largeのロード

初回実行時にモデル（約300MBほど）が自動ダウンロードされます。


In [ ]:
from funasr import AutoModel

emotion_model = AutoModel(
    model="iic/emotion2vec_plus_large",
    hub="hf",  # 日本国外からのアクセスは "hf"（Hugging Face経由）を指定
)

print("emotion2vec+largeのロードが完了しました。")


## 3. JVNVコーパスの読み込みとサンプリング

`style`列が感情ラベル（anger, disgust, fear, happy, sad, surprise）、
`speaker_id`が話者（4名）、`session`が収録セッション（regular / free）です。

感情ごとに一定数をランダムサンプリングして音声ファイルとして保存します。


In [ ]:
from datasets import load_dataset

jvnv = load_dataset("asahi417/jvnv-emotional-speech-corpus", split="test")
print("総クリップ数:", len(jvnv))

meta = jvnv.remove_columns(["audio"]).to_pandas()
print(meta["style"].value_counts())
print(meta["speaker_id"].value_counts())


In [ ]:
N_PER_EMOTION = 30  # 感情ごとのサンプル数（実行時間に応じて調整してください）
RANDOM_SEED = 42

# 注意: meta.groupby("style").apply(lambda g: g.sample(...)) は使わない。
# pandas 2.2以降、groupby().apply()はグループ化に使った列(ここでは"style")を
# 関数に渡す前にデータフレームから除外するようになったため、
# 結果に"style"列が含まれずKeyErrorになる場合がある。
# DataFrameGroupBy.sample()を直接使えばこの問題を避けられる。
sampled = meta.groupby("style", group_keys=False).sample(
    n=N_PER_EMOTION, random_state=RANDOM_SEED
)
print(f"サンプリングしたクリップ数: {len(sampled)}")
sampled["style"].value_counts()


In [ ]:
jvnv_selected = jvnv.select(sampled.index.tolist())

sample_paths = []
for row_meta, item in zip(sampled.itertuples(), jvnv_selected):
    out_path = f"audio_samples/jvnv_{row_meta.Index}_{row_meta.style}.wav"
    audio_array = np.asarray(item["audio"]["array"], dtype=np.float32)
    sr = item["audio"]["sampling_rate"]
    sf.write(out_path, audio_array, sr)
    sample_paths.append({
        "true_emotion": row_meta.style,
        "speaker_id": row_meta.speaker_id,
        "session": row_meta.session,
        "path": out_path,
    })

print(f"{len(sample_paths)}件の音声を保存しました。")


## 4. 推論の実行

In [ ]:
# JVNVのラベル -> emotion2vec+の英語ラベル対応
# (emotion2vec+の "disgusted"/"fearful"/"surprised" は形容詞形なので読み替える)
E2V_LABEL_NORMALIZE = {
    "angry": "anger",
    "disgusted": "disgust",
    "fearful": "fear",
    "happy": "happy",
    "neutral": "neutral",
    "other": "other",
    "sad": "sad",
    "surprised": "surprise",
    "unknown": "unknown",
}


def predict_emotion(path):
    res = emotion_model.generate(path, granularity="utterance", extract_embedding=False)
    labels = res[0]["labels"]
    scores = res[0]["scores"]
    # ラベルは "生気/angry" のような "現地語/英語" 形式の場合があるため、英語部分だけを取り出す
    labels = [l.split("/")[-1] if "/" in l else l for l in labels]
    best_idx = int(np.argmax(scores))
    return {
        "predicted_raw": labels[best_idx],
        "predicted": E2V_LABEL_NORMALIZE.get(labels[best_idx], labels[best_idx]),
        "confidence": float(scores[best_idx]),
        "all_labels": labels,
        "all_scores": scores,
    }


rows = []
for item in sample_paths:
    pred = predict_emotion(item["path"])
    rows.append({
        "true_emotion": item["true_emotion"],
        "speaker_id": item["speaker_id"],
        "session": item["session"],
        "path": item["path"],
        "predicted_emotion": pred["predicted"],
        "confidence": pred["confidence"],
    })

df_results = pd.DataFrame(rows)
df_results.head(10)


## 5. Accuracyと混同行列

JVNVの6感情（anger, disgust, fear, happy, sad, surprise）を正解ラベルとし、
emotion2vec+の予測トップ1ラベルと突き合わせます。


In [ ]:
overall_accuracy = accuracy_score(df_results["true_emotion"], df_results["predicted_emotion"])
print(f"全体Accuracy: {overall_accuracy:.1%}")
print()
print(classification_report(df_results["true_emotion"], df_results["predicted_emotion"], zero_division=0))


In [ ]:
EMOTIONS_ORDER = ["anger", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
present_labels = [e for e in EMOTIONS_ORDER if e in set(df_results["true_emotion"]) | set(df_results["predicted_emotion"])]

cm = confusion_matrix(df_results["true_emotion"], df_results["predicted_emotion"], labels=present_labels)
cm_normalized = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm_normalized, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(present_labels)))
ax.set_yticks(range(len(present_labels)))
ax.set_xticklabels(present_labels, rotation=45, ha="right")
ax.set_yticklabels(present_labels)
ax.set_xlabel("emotion2vec+ の予測")
ax.set_ylabel("JVNVの正解ラベル")
ax.set_title(f"混同行列（正規化済み、全体Accuracy={overall_accuracy:.1%}）")

for i in range(len(present_labels)):
    for j in range(len(present_labels)):
        val = cm_normalized[i, j]
        color = "white" if val > 0.5 else "black"
        ax.text(j, i, f"{val:.0%}\n({cm[i, j]})", ha="center", va="center", color=color, fontsize=9)

fig.colorbar(im, ax=ax, label="行方向（正解）に対する割合")
plt.tight_layout()
plt.savefig("results/jvnv_emotion2vec_confusion_matrix.png", dpi=150)
plt.show()


## 6. 感情ごとのAccuracy

In [ ]:
# 注意: df_results.groupby("true_emotion").apply(lambda g: g["true_emotion"]...)は使わない。
# pandas 2.2以降、groupby().apply()はグループ化に使った列("true_emotion")を
# 関数に渡す前にデータフレームから除外するため、g["true_emotion"]がKeyErrorになる。
# 先に正解/不正解の列を作ってからgroupbyする方式なら、この問題を回避できる。
df_results["correct"] = df_results["true_emotion"] == df_results["predicted_emotion"]
per_emotion_acc = df_results.groupby("true_emotion")["correct"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(per_emotion_acc.index, per_emotion_acc.values, color="#4C72B0", edgecolor="black")
ax.axhline(overall_accuracy, color="gray", linestyle="--", label=f"全体平均 ({overall_accuracy:.1%})")
ax.set_ylim(0, 1)
ax.set_ylabel("Accuracy")
ax.set_title("感情ごとのAccuracy（JVNV、emotion2vec+large、ファインチューニングなし）")
for bar, val in zip(bars, per_emotion_acc.values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.02, f"{val:.0%}", ha="center")
ax.legend()
plt.tight_layout()
plt.savefig("results/jvnv_emotion2vec_per_emotion_accuracy.png", dpi=150)
plt.show()


## 7. ネガティブ検出（二値分類）としての評価

サポート業務などの実務では、happyを検出できるかどうかより、怒っている・困っている顧客を
確実に検出できるかが重要になるケースが多くあります。エスカレーションや対応優先度の判断に
直結するのは、多くの場合ネガティブな感情の検出だからです。

そこで、anger・disgust・fear・sadを「ネガティブ」、happy・surpriseを「非ネガティブ」として
二値分類の指標（Recall・Precision・Accuracy）を算出します。6クラスでの評価とは異なり、
fear↔sadのような「ネガティブ同士の取り違え」は誤りとしてカウントされません。


In [ ]:
NEGATIVE_EMOTIONS = {"anger", "disgust", "fear", "sad"}

df_results["true_negative"] = df_results["true_emotion"].isin(NEGATIVE_EMOTIONS)
df_results["pred_negative"] = df_results["predicted_emotion"].isin(NEGATIVE_EMOTIONS)

tp = ((df_results["true_negative"]) & (df_results["pred_negative"])).sum()   # 正しくネガティブと検出
fn = ((df_results["true_negative"]) & (~df_results["pred_negative"])).sum()  # 見逃し
fp = ((~df_results["true_negative"]) & (df_results["pred_negative"])).sum()  # 誤検知
tn = ((~df_results["true_negative"]) & (~df_results["pred_negative"])).sum() # 正しく非ネガと判定

recall = tp / (tp + fn)
precision = tp / (tp + fp)
binary_accuracy = (tp + tn) / len(df_results)

print(f"ネガティブ発話: {df_results['true_negative'].sum()}件 / 非ネガティブ発話: {(~df_results['true_negative']).sum()}件")
print()
print(f"TP(正しく検出): {tp}  FN(見逃し): {fn}  FP(誤検知): {fp}  TN(正しく非ネガ): {tn}")
print()
print(f"再現率(Recall): {recall:.1%}")
print(f"適合率(Precision): {precision:.1%}")
print(f"二値Accuracy: {binary_accuracy:.1%}")


In [ ]:
# 見逃し・誤検知の内訳（どの感情がどう誤判定されたか）
missed = df_results[(df_results["true_negative"]) & (~df_results["pred_negative"])]
false_alarm = df_results[(~df_results["true_negative"]) & (df_results["pred_negative"])]

print("見逃しケースの内訳（正解emotion -> 誤predicted）:")
print(missed.groupby(["true_emotion", "predicted_emotion"]).size())
print()
print("誤検知ケースの内訳（正解emotion -> 誤predicted）:")
print(false_alarm.groupby(["true_emotion", "predicted_emotion"]).size())


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
metrics = {"Recall": recall, "Precision": precision, "Accuracy": binary_accuracy}
bars = ax.bar(metrics.keys(), metrics.values(), color="#4C72B0", edgecolor="black")
ax.set_ylim(0, 1)
ax.set_ylabel("スコア")
ax.set_title("ネガティブ検出（二値分類）の評価指標")
for bar, val in zip(bars, metrics.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.02, f"{val:.1%}", ha="center")
plt.tight_layout()
plt.savefig("results/jvnv_emotion2vec_binary_metrics.png", dpi=150)
plt.show()


## 8. 結果のCSV保存


In [ ]:
csv_path = "results/jvnv_emotion2vec_results.csv"
df_results.to_csv(csv_path, index=False)
print(f"結果を保存しました: {csv_path}")
df_results


## まとめ

- **全体Accuracy**が、実務で使える目安（例えばランダム推測を大きく上回るか、業務要件の閾値を
  超えるか）に達しているかを確認してください
- **混同行列**で、どの感情とどの感情が混同されやすいかを確認してください
  （例：angryとsurpriseの混同、fearとsadの混同など、音響的に近い感情はモデルによらず
  混同されやすい傾向があります）
- ここまでの英語VAD実験と違い、**ファインチューニングなしで日本語の実発話に対して定量的な
  Accuracyが出せた**こと自体が、実務者にとって重要な情報です。この数値をもとに、
  JVNVでのファインチューニングが本当に必要かどうかを判断できます
